# Preview `te_process.csv`

Load and print the first 10 rows. Later cells scan the full CSV for unique-value counts and min/max ranges.

In [5]:
from pathlib import Path

import pandas as pd

print("working")
csv_path = Path("te_process.csv")
df = pd.read_csv(csv_path, nrows=10)
pd.set_option("display.max_columns", None)
print(df)


working
   faultNumber  simulationRun  sample  xmeas_1  xmeas_2  xmeas_3  xmeas_4  \
0          0.0            1.0       1  0.25038   3674.0   4529.0   9.2320   
1          0.0            1.0       2  0.25109   3659.4   4556.6   9.4264   
2          0.0            1.0       3  0.25038   3660.3   4477.8   9.4426   
3          0.0            1.0       4  0.24977   3661.3   4512.1   9.4776   
4          0.0            1.0       5  0.29405   3679.0   4497.0   9.3381   
5          0.0            1.0       6  0.29303   3691.7   4502.2   9.3780   
6          0.0            1.0       7  0.24301   3658.8   4541.6   9.3374   
7          0.0            1.0       8  0.24090   3653.3   4500.0   9.3495   
8          0.0            1.0       9  0.29416   3654.3   4454.7   9.3213   
9          0.0            1.0      10  0.29372   3675.9   4487.4   9.4107   

   xmeas_5  xmeas_6  xmeas_7  xmeas_8  xmeas_9  xmeas_10  xmeas_11  xmeas_12  \
0   26.889   42.402   2704.3   74.863   120.41   0.33818    80.0

In [6]:
print("=== unique count per column (full CSV) ===")
uniques = {}
for chunk in pd.read_csv(csv_path, chunksize=200_000):
    for col in chunk.columns:
        uniques.setdefault(col, set()).update(chunk[col].dropna().unique())

for col, values in uniques.items():
    print(f"{col}: {len(values)}")


=== unique count per column (full CSV) ===
faultNumber: 21
simulationRun: 500
sample: 960
xmeas_1: 305017
xmeas_2: 4807
xmeas_3: 14017
xmeas_4: 29636
xmeas_5: 2820
xmeas_6: 4217
xmeas_7: 5538
xmeas_8: 21262
xmeas_9: 139
xmeas_10: 68547
xmeas_11: 15751
xmeas_12: 8353
xmeas_13: 5895
xmeas_14: 10505
xmeas_15: 8640
xmeas_16: 5635
xmeas_17: 6261
xmeas_18: 20038
xmeas_19: 122563
xmeas_20: 13403
xmeas_21: 18924
xmeas_22: 18426
xmeas_23: 15548
xmeas_24: 24202
xmeas_25: 16932
xmeas_26: 15055
xmeas_27: 12863
xmeas_28: 18207
xmeas_29: 22835
xmeas_30: 3393
xmeas_31: 24985
xmeas_32: 63015
xmeas_33: 17538
xmeas_34: 15231
xmeas_35: 28172
xmeas_36: 15730
xmeas_37: 223331
xmeas_38: 61903
xmeas_39: 62907
xmeas_40: 5453
xmeas_41: 5705
xmv_1: 45344
xmv_2: 67098
xmv_3: 230849
xmv_4: 63452
xmv_5: 97020
xmv_6: 189224
xmv_7: 23169
xmv_8: 18990
xmv_9: 184310
xmv_10: 193879
xmv_11: 133491
source: 2
fault_status: 2


In [7]:
print("=== min / max per column (full CSV) ===")
mins = None
maxs = None
for chunk in pd.read_csv(csv_path, chunksize=200_000):
    chunk_min = chunk.min()
    chunk_max = chunk.max()
    if mins is None:
        mins, maxs = chunk_min, chunk_max
    else:
        mins = mins.combine(chunk_min, min)
        maxs = maxs.combine(chunk_max, max)

for col in mins.index:
    print(f"{col}: min={mins[col]}, max={maxs[col]}")


=== min / max per column (full CSV) ===
faultNumber: min=0.0, max=20.0
simulationRun: min=1.0, max=500.0
sample: min=1, max=960
xmeas_1: min=-0.0049855, max=1.0175
xmeas_2: min=3308.4, max=3906.7
xmeas_3: min=3540.7, max=5175.8
xmeas_4: min=6.6399, max=12.24
xmeas_5: min=25.348, max=28.565
xmeas_6: min=39.656, max=44.653
xmeas_7: min=2413.8, max=3000.5
xmeas_8: min=61.132, max=87.189
xmeas_9: min=119.61, max=121.01
xmeas_10: min=0.018396, max=0.82073
xmeas_11: min=68.097, max=87.591
xmeas_12: min=44.627, max=55.481
xmeas_13: min=2317.1, max=2945.4
xmeas_14: min=18.436, max=33.092
xmeas_15: min=44.311, max=55.912
xmeas_16: min=2870.4, max=3452.7
xmeas_17: min=19.137, max=27.239
xmeas_18: min=52.119, max=74.699
xmeas_19: min=-3.5372, max=466.71
xmeas_20: min=230.15, max=400.71
xmeas_21: min=79.898, max=100.28
xmeas_22: min=62.636, max=83.808
xmeas_23: min=23.225, max=40.211
xmeas_24: min=7.4438, max=10.345
xmeas_25: min=16.909, max=36.469
xmeas_26: min=5.8989, max=7.8854
xmeas_27: min=12

In [8]:
n_lines = 0
with csv_path.open("rb") as f:
    for buf in iter(lambda: f.read(1024 * 1024), b""):
        n_lines += buf.count(b"\n")

n_rows = n_lines - 1  # subtract header
print(f"rows: {n_rows}")


rows: 15330000
